# Main Code for NYTC Qualification
## 2nd Challenge

1) Robot recognizes left or right. 
2) Robot moves to the path accordingly
3) Navigates the rest with line tracing

## Setup

In [44]:
import importlib
import cv2
import numpy as np
from ugot import ugot
import pose_yolo
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

# Reload the pose control module so any edits to pose_yolo.py take effect
# without restarting the kernel.


# Create the robot controller object
got = ugot.UGOT()

# Connect to the UGOT robot using its IP address.
# Change this string if your robot is on a different IP.
got.initialize("192.168.88.1")

# Load the built-in computer vision models that will be used later in the notebook.
# You can load additional models here if needed — just add the model name to the list.
got.load_models(
    [
        "color_recognition",  # detects dominant colors 
        "word_recognition",  # OCR: reads printed text
        "line_recognition",  # for line-following tasks
        "face_recognition",  # identifies registered faces by name
        "apriltag_qrcode" # AprilTag recognition
    ]
)

# Select the default line-tracking mode.
# 0 = single-line mode (follows one line at a time)
got.set_track_recognition_line(0)

# Open the camera stream so later functions can read live frames
got.open_camera()

print("Done!")

192.168.88.1:50051
Done!


## Line Tracing & Text Recognition

### Function Initialization

In [45]:
def line_follow(mult=0.25, speed=35):
    """Follow the detected line by turning proportionally to the line offset."""
    # Read line-tracking information from the robot.
    # `offset` tells how far the detected line is from the center.
    # `type` describes the line/intersection pattern detected.
    # `x` and `y` are the detected line position coordinates.
    offset, line_type, x, y = got.get_single_track_total_info()

    # Convert the line offset into a turning speed.
    # Larger offset -> stronger rotation to re-center on the line.
    rotation_speed = int(offset * mult)

    # Move forward while rotating to stay aligned with the line.
    got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=rotation_speed)

    # Return the detection info so the main program can make decisions.
    return line_type, x, y

### Line Tracing

In [49]:
try:
    while True:
        line_type, _, _ = line_follow(mult=0.25, speed=15)
        if line_type in [0,2]:
            print("can not see path")
            got.mecanum_stop()
            break
    got.mecanum_translate_speed_times(angle=0, speed=20, times=13, unit=1)
    # move forward


    print("Run more")
    while True:
        # Read any text currently visible in the camera frame.
        text = got.get_words_result()

        print("Text:", text)

        # React to specific command words
            # Turn counter-clockwise by ~45 degrees
        if text in ["LEFT", "RIGHT"]:
            turn = 2
            if(text == "RIGHT"): turn = 3
            while True:
                got.mecanum_turn_speed_times(turn=turn, speed=30, times=20, unit=2)
                line_type, _, _ = line_follow(mult=0.25, speed=0)
                if line_type == 1:
                    print("Turned and see line")
                    break
            break

    while True:
        line_type, _, _ = line_follow(mult=0.25, speed=15)
        if line_type in [0]:
            print("can not see path")
            break   

    got.mecanum_stop() 

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")

can not see path
Run more
Text: 
Text: 
Text: 
Text: 
Text: 
Text: 
Text: 
Text: 
Text: 
Text: 
Text: 
Text: 
Text: 
Text: 
Text: 
Text: 
Text: RIGHT
Turned and see line
can not see path
